# Oil Price Prediction — End-to-End Pipeline

This notebook runs the full pipeline end-to-end:

1. Load and explore the raw data (WTI, GPRD, GEPU).
2. Build the processed dataset (`data/processed/`).
3. Train three families of models: baseline, deep learning, TFT.
4. Compare them on the test set.
5. Preview the FastAPI deployment.

**Target:** next-day WTI crude oil spot price (USD / barrel).

**Window:** 1997-01-01 → 2025-11-30 (limited by EPU coverage).

**Splits (chronological, no shuffle):**

- Train ≤ 2020-12-31
- Val   2021-01-01 → 2022-12-31
- Test  2023-01-01 → end

## 1. Setup & Imports

In [ ]:
import sys, os, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
%matplotlib inline

from src.data.preprocessing import load_wti, load_gpr, load_epu, merge_sources, add_features, add_target

## 2. Data Loading & EDA

In [ ]:
RAW = PROJECT_ROOT / 'data' / 'raw'
wti = load_wti(RAW / 'RWTCd.xls')
gpr = load_gpr(RAW / 'data_gpr_daily_recent.xls')
epu = load_epu(RAW / 'Global_Policy_Uncertainty_Data.xlsx')

print('WTI:', wti.shape, '| GPR:', gpr.shape, '| EPU:', epu.shape)
wti.head()

In [ ]:
display(gpr.head())
display(epu.head())
display(wti.describe())

In [ ]:
events = {
    '2008-09-15': 'Lehman / GFC',
    '2014-11-27': 'OPEC keeps quota',
    '2020-04-20': 'COVID / negative WTI',
    '2022-02-24': 'Russia invades Ukraine',
}
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(wti['date'], wti['WTI_price'], color='#1f77b4')
for d, label in events.items():
    ts = pd.Timestamp(d)
    ax.axvline(ts, color='red', alpha=0.4, linestyle='--')
    ax.annotate(label, xy=(ts, ax.get_ylim()[1]*0.9), rotation=30, fontsize=9)
ax.set_title('WTI spot price with key geopolitical events')
ax.set_xlabel('Date'); ax.set_ylabel('USD / barrel')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
axes[0].plot(gpr['date'], gpr['GPRD'], color='#d62728')
axes[0].set_title('Geopolitical Risk Index (GPRD)')
axes[0].set_ylabel('GPRD')

epu_dates = pd.to_datetime(epu['Year'].astype(str) + '-' + epu['Month'].astype(str) + '-01')
axes[1].plot(epu_dates, epu['GEPU_current'], color='#2ca02c')
axes[1].set_title('Global Economic Policy Uncertainty (GEPU current)')
axes[1].set_ylabel('GEPU'); axes[1].set_xlabel('Date')
plt.tight_layout(); plt.show()

In [ ]:
merged = merge_sources(wti, gpr, epu)
print('Merged shape:', merged.shape)
print('Date range :', merged['date'].min().date(), '->', merged['date'].max().date())
print('Missing:', merged.isna().sum().to_dict())
merged.head()

In [ ]:
corr = merged.select_dtypes('number').corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlation heatmap'); plt.tight_layout(); plt.show()

## 3. Build the processed dataset

In [ ]:
from src.data.build_dataset import build_dataset
splits = build_dataset(
    data_dir=PROJECT_ROOT/'data'/'raw',
    processed_dir=PROJECT_ROOT/'data'/'processed',
    model_dir=PROJECT_ROOT/'models',
)

## 4. Feature analysis — lag, ACF/PACF, stationarity

In [ ]:
full = pd.read_csv(PROJECT_ROOT/'data'/'processed'/'full_dataset.csv', parse_dates=['date'])
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(full['WTI_price'].shift(1), full['WTI_price'], s=4, alpha=0.4)
ax.set_title('Lag-1 plot — WTI_price vs WTI_price.shift(1)')
ax.set_xlabel('WTI_t-1'); ax.set_ylabel('WTI_t')
plt.tight_layout(); plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

fig, axes = plt.subplots(2, 1, figsize=(11, 6))
plot_acf(full['WTI_price'].dropna(), ax=axes[0], lags=40)
plot_pacf(full['WTI_price'].dropna(), ax=axes[1], lags=40, method='ywm')
plt.tight_layout(); plt.show()

adf = adfuller(full['WTI_price'].dropna())
print(f'ADF stat: {adf[0]:.3f}  p-value: {adf[1]:.4f}')
ret = np.log(full['WTI_price']).diff().dropna()
adf_ret = adfuller(ret)
print(f'ADF on log returns: {adf_ret[0]:.3f}  p-value: {adf_ret[1]:.4f}')

## 5. Model 1 — Baseline (naive + Random Forest)

In [ ]:
from src.training.train_baseline import train_and_evaluate
baseline_metrics = train_and_evaluate(
    processed_dir=PROJECT_ROOT/'data'/'processed',
    model_dir=PROJECT_ROOT/'models',
    metrics_dir=PROJECT_ROOT/'outputs'/'metrics',
    figures_dir=PROJECT_ROOT/'outputs'/'figures',
)
baseline_metrics

In [ ]:
from IPython.display import Image
Image(filename=str(PROJECT_ROOT/'outputs'/'figures'/'baseline_predictions.png'))

## 6. Model 2 — Deep Learning (BiLSTM + CNN/GRU)

These cells run shorter training (20 epochs) so the notebook completes quickly. For
production-quality results, use the CLI: `python -m src.training.train_deep --model bilstm --epochs 100`.

In [ ]:
from src.training.train_deep import train as train_deep
bilstm = train_deep('bilstm',
    processed_dir=PROJECT_ROOT/'data'/'processed',
    model_dir=PROJECT_ROOT/'models',
    metrics_dir=PROJECT_ROOT/'outputs'/'metrics',
    figures_dir=PROJECT_ROOT/'outputs'/'figures',
    epochs=20)
bilstm['test']

In [ ]:
cnn_gru = train_deep('cnn_gru',
    processed_dir=PROJECT_ROOT/'data'/'processed',
    model_dir=PROJECT_ROOT/'models',
    metrics_dir=PROJECT_ROOT/'outputs'/'metrics',
    figures_dir=PROJECT_ROOT/'outputs'/'figures',
    epochs=20)
cnn_gru['test']

In [ ]:
for name in ('bilstm','cnn_gru'):
    for fig in ('loss_curve','predictions','residuals'):
        print(name, fig)
        display(Image(filename=str(PROJECT_ROOT/'outputs'/'figures'/f'deep_{name}_{fig}.png')))

## 7. Model 3 — Advanced TFT (with q10 / q50 / q90)

In [ ]:
from src.training.train_advanced import train as train_tft
tft = train_tft(
    processed_dir=PROJECT_ROOT/'data'/'processed',
    model_dir=PROJECT_ROOT/'models',
    metrics_dir=PROJECT_ROOT/'outputs'/'metrics',
    figures_dir=PROJECT_ROOT/'outputs'/'figures',
    epochs=20)
tft['test']

In [ ]:
display(Image(filename=str(PROJECT_ROOT/'outputs'/'figures'/'advanced_tft_loss_curve.png')))
display(Image(filename=str(PROJECT_ROOT/'outputs'/'figures'/'advanced_tft_predictions_with_uncertainty.png')))
display(Image(filename=str(PROJECT_ROOT/'outputs'/'figures'/'advanced_tft_feature_importance.png')))

## 8. Cross-model comparison

In [ ]:
from src.evaluation.compare_models import run_comparison
df_cmp = run_comparison(
    processed_dir=PROJECT_ROOT/'data'/'processed',
    model_dir=PROJECT_ROOT/'models',
    metrics_dir=PROJECT_ROOT/'outputs'/'metrics',
    figures_dir=PROJECT_ROOT/'outputs'/'figures',
    output_md=PROJECT_ROOT/'outputs'/'model_analysis.md',
)
df_cmp

In [ ]:
display(Image(filename=str(PROJECT_ROOT/'outputs'/'figures'/'model_comparison.png')))
display(Image(filename=str(PROJECT_ROOT/'outputs'/'figures'/'all_predictions_overlay.png')))

## 9. Deployment preview — calling the FastAPI endpoint

Start the API in a separate terminal:

```bash
uvicorn src.app.api:app --host 0.0.0.0 --port 8000
```

Then run the cell below to query it.

In [ ]:
import requests

API_URL = os.environ.get('API_URL', 'http://localhost:8000')

try:
    health = requests.get(f'{API_URL}/health', timeout=3).json()
    print('Health:', health)
    feat = health.get('n_features', 0)
    seq_len = health.get('seq_len', 30)
    test_df = pd.read_csv(PROJECT_ROOT/'data'/'processed'/'test.csv', parse_dates=['date'])
    feat_cols = json.loads((PROJECT_ROOT/'data'/'processed'/'feature_names.json').read_text())
    feat_cols = [c for c in feat_cols if c in test_df.columns]
    seq = test_df[feat_cols].tail(seq_len).values.tolist()
    for model in ('naive','rf','bilstm','cnn_gru','tft'):
        r = requests.post(f'{API_URL}/predict', json={'model': model, 'sequence': seq}, timeout=10)
        print(model, '->', r.json())
except Exception as e:
    print('API not reachable:', e)

## 10. Conclusion

Refer to `outputs/model_analysis.md` for the written commentary.

Key takeaways:

- Naive persistence is a hard reference to beat on day-to-day price.
- The TFT trades a small accuracy hit for calibrated uncertainty (q10–q90) and interpretable feature weights.
- For production, deploy the model with the best MAE / inference-cost trade-off, or an ensemble of the deep models.

Source data:

- WTI Spot Price — EIA
- Geopolitical Risk Index — Caldara & Iacoviello (2022)
- Global Economic Policy Uncertainty — Davis (2016)